<a href="https://colab.research.google.com/github/anishbosegupta/Pfizer-Supply-Chain-Document-Processing/blob/main/Pharmaceutical_Document_Q%26A_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Pharmaceutical Document Q&A System with Intelligent RAG Pipeline

This comprehensive notebook, demonstrates how to build a production-ready Question Answering (Q&A) system specifically tailored for the complexities of pharmaceutical documentation. In an industry where accuracy, traceability, and compliance are paramount, quickly and reliably extracting information from vast repositories of documents like Certificates of Quality, Packaging Specifications, and regulatory filings is crucial.

Traditional search methods often fall short, struggling with the nuanced, multi-document nature of these critical records. This notebook presents an **Intelligent Retrieval-Augmented Generation (RAG) Pipeline** designed to overcome these challenges by combining cutting-edge AI techniques:

*  **Intelligent Multi-Document Detection & Classification:** Automatically identifies and categorizes distinct document types within a single "blob" PDF, such as a 'Cover Letter' or 'Certificate of Quality', ensuring contextually relevant processing.
*  **Query Routing for Enhanced Retrieval Accuracy:** Dynamically directs user queries to the most appropriate document type for superior precision, preventing irrelevant information from diluting search results.
*  **Rich Metadata Preservation & Attribution:** Every piece of information retrieved is meticulously linked back to its original source (document type, page range), crucial for auditability and compliance in pharmaceutical contexts.
*  **Robust Chunking & Vector Indexing:** Employs advanced techniques to break down documents into semantically meaningful chunks and create efficient vector embeddings for lightning-fast, relevant information retrieval.
*  **User-Friendly Gradio Interface:** Provides an intuitive web interface for seamless document upload, processing, and natural language Q&A, making powerful AI accessible to quality engineers, regulatory specialists, and researchers.

By leveraging this system, you can dramatically improve the efficiency and accuracy of information retrieval from your pharmaceutical documents, ensuring critical decisions are based on verifiable and precise data. Follow along to deploy your own intelligent Q&A assistant!

## 📚 Setup and Installation
Install all necessary packages

In [ ]:
# ============================================
# STEP 1: Install Required Packages
# ============================================
# WHAT IS HAPPENING:
# Installing all Python libraries needed for the pharmaceutical document
# Q&A pipeline: Gradio for the UI, PyMuPDF for PDF parsing, FAISS for
# vector search, Sentence Transformers for embeddings, and LlamaIndex for advanced document processing.
#
# WHY THIS MATTERS:
# Each library handles a distinct part of the pipeline. Without them, we
# cannot extract text from PDFs, generate vector embeddings, or query the
# Mistral model. The -q flag suppresses verbose output so the cell is easier
# to read.
# ============================================
# Install required packages
!pip install -q gradio
!pip install -q gradio_pdf
!pip install -q pypdf PyPDF2 pymupdf
!pip install -q sentence-transformers transformers
!pip install -q faiss-cpu
!pip install -q numpy pandas
!pip install -q pytesseract Pillow

# Install LlamaIndex packages for enhanced document processing
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-faiss
!pip install -q llama-index-llms-llama-cpp
!pip install -q torch

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Check CUDA version
!nvcc --version

In [ ]:
# Check CUDA version first
!nvcc --version

# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

## 🔧 Core Imports and Configuration

In [ ]:
# ============================================
# STEP 2: Core Imports and Configuration
# ============================================
# WHAT IS HAPPENING:
# Importing all libraries, loading the Mistral-7B-Instruct model locally
# via LlamaCPP (no API key required), and loading the all-MiniLM-L6-v2
# embedding model that will convert text into vector representations.
#
# WHY THIS MATTERS:
# The Mistral model powers document classification and answer generation,
# running fully locally using a quantized GGUF file downloaded from
# HuggingFace. The embedding model converts text chunks into numerical
# vectors so we can perform semantic similarity search with FAISS.
# Unlike Gemini, no API key or Colab Secrets are needed — the model
# runs on-device using llama.cpp.
# ==================================================================
import gradio as gr
from gradio_pdf import PDF
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import json
from datetime import datetime
import hashlib

# LlamaIndex imports for enhanced document processing
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core import Settings
import os
from google.colab import userdata

MODEL_PATH = "mistral.gguf"

#Download the Mistral model
if not os.path.exists(MODEL_PATH):
    os.system(
        'wget -q https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O mistral.gguf'
    )

#This code converts a list of chat messages, accounting for system, user, and assistant roles,
#into a single, formatted string prompt suitable for an instruction-tuned language model like Mistral.
def messages_to_prompt(messages):
    prompt = ""
    system_text = ""
    for m in messages:
        role = m.role.value if hasattr(m.role, 'value') else str(m.role)
        if role == "system":
            system_text = m.content + "\n\n"
        elif role == "user":
            prompt += f"[INST] {system_text}{m.content} [/INST]"
            system_text = ""
        elif role == "assistant":
            prompt += f" {m.content}</s>"
    return prompt

#This code formats a given completion into a specific prompt string that instructs a language model to answer
#only using provided context and cite its sources.
def completion_to_prompt(completion):
    return f"[INST] Answer ONLY using provided context. Cite sources.\n\n{completion} [/INST]"

#This code initializes a LlamaCPP language model with specified parameters such as model path,
#temperature, token limits, and custom functions for formatting prompts from messages and completions.
llm = LlamaCPP(
    model_path=MODEL_PATH,
    temperature=0.1,
    max_new_tokens=256,
    context_window=4096,
    model_kwargs={"n_gpu_layers": -1, "n_threads": 4},
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
)

Settings.llm = llm

# Initialize embedding models (both for compatibility)
llama_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = llama_embed_model

print("Imports and configuration complete.")

## 📄 Data Structures for Enhanced Document Management

In [ ]:
# ============================================
# STEP 3: Data Structures for Document Management
# ============================================
#
# WHAT IS HAPPENING:
# Defining three Python dataclasses that act as structured containers for
# the information extracted from the PDF: individual page data, logical
# document groupings, and rich chunk metadata.
#
# WHY THIS MATTERS:
# A pharmaceutical blob PDF may contain many distinct documents -- a Cover
# Letter, a Certificate of Quality, a Packaging Specification, etc. -- all
# merged into one file. These dataclasses let us track which pages belong
# to which document and carry that metadata through the entire pipeline so
# answers can cite exact page ranges and document types.
# ============================================

@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = None

@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

## 🧠 Document Intelligence Functions

In [ ]:
# ============================================
# STEP 4: Document Intelligence Functions
# ============================================
#
# WHAT IS HAPPENING:
# Defining two key functions: classify_document_type (uses Mistral-7B Instruct model to
# identify which of the 7 pharmaceutical document types a page belongs to)
# and detect_document_boundary.
# A helper clean_doc_type() normalises the LLM's free-text response into
# one of the valid labels.
#
# WHY THIS MATTERS:
# A pharmaceutical blob PDF is a single file that bundles many different
# document types together. Before we can answer questions accurately, we
# must split it into logical segments so retrieval can be scoped to the
# right document. For example, a question about lot numbers should pull
# from the Certificate of Quality, not the Cover Letter.
# ============================================

VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

def clean_doc_type(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').replace('.', '').strip().lower()

    # Explicit keyword matching — order matters, most specific first
    if 'material' in cleaned and 'description' in cleaned:
        return 'Material Description'
    if 'bse' in cleaned or 'tse' in cleaned:
        return 'Bse/Tse Declaration'
    if 'chain' in cleaned and 'custody' in cleaned:
        return 'Chain Of Custody'
    if 'certificate' in cleaned or 'quality' in cleaned:
        return 'Certificate Of Quality'
    if 'packaging' in cleaned or 'specification' in cleaned:
        return 'Packaging Specification'
    if 'supplier' in cleaned or 'qualification' in cleaned:
        return 'Supplier Qualification'
    if 'cover' in cleaned or 'letter' in cleaned:
        return 'Cover Letter'

    # Fallback — try exact label matching
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned:
            return label

    return 'Other'


def classify_document_type(text: str, max_length: int = 1500) -> str:
    text_sample = text[:max_length] if len(text) > max_length else text

    prompt = f"""You are a pharmaceutical document classifier. Classify the page below into EXACTLY ONE type.

CLASSIFICATION RULES — look for these specific keywords:
- Cover Letter: contains "To Whom It May Concern", "Dear", "storage conditions", "sincerely"
- Certificate Of Quality: contains "lot number", "batch", "expiration date", "manufacture date", "test results", "autoclave", "gamma"
- Packaging Specification: contains "part number", "PKG-", "packaging component", "configuration", "change history", "blister", "carton"
- Bse/Tse Declaration: contains "BSE", "TSE", "bovine", "animal origin", "spongiform", "transmissible"
- Material Description: contains "materials of construction", "sterilization compatibility", "physical properties", "polymer", "resin", "durometer"
- Supplier Qualification: contains "ISO 9001", "ISO 13485", "audit", "supplier", "certification", "approved supplier"
- Chain Of Custody: contains "manufactured assembly", "traceability", "shipment", "chain of custody", "manufacturing flow"
- Other: does not match any above

Page content:
{text_sample}

Respond with ONLY the document type name from the list above. No explanation. No punctuation."""

    try:
        response = str(llm.complete(prompt)).strip()
        print(f"[CLASSIFY] Raw response: '{response[:80]}'")
        return clean_doc_type(response)
    except Exception as e:
        print(f"Classification error: {e}")
        return 'Other'

def detect_document_boundary(prev_text: str, curr_text: str,
                            current_doc_type: str = None,
                            curr_doc_type: str = None) -> bool:  # ← add curr_doc_type param
    if not prev_text or not curr_text:
        return False

    # ── KEY FIX: if classification already says different type, trust it ──
    if curr_doc_type and current_doc_type and curr_doc_type != current_doc_type:
        if curr_doc_type != "Other":  # don't trust "Other" classifications
            print(f"  [BOUNDARY OVERRIDE] Type changed: {current_doc_type} → {curr_doc_type}, forcing new document")
            return False  # new document

    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text

    prompt = f"""Determine if these two pages are from the SAME pharmaceutical document.

Current document type: {current_doc_type or 'Unknown'}

A NEW document starts when the page has:
- A different document title or heading (e.g., "Certificate of Quality"
  vs "Packaging Specification" vs "Material Description Sheet")
- A completely different topic or subject matter
- Its own header with a new document number or reference

Pages belong to the SAME document when:
- The second page says "continued" or "page 2 of 2"
- The content directly continues the previous page's discussion
- They share the same document number or title

End of Previous Page:
...{prev_sample}

Start of Current Page:
{curr_sample}...

Answer ONLY 'Yes' if same document or 'No' if different document."""

    try:
        response = str(llm.complete(prompt)).strip().lower()
        print(f"  [BOUNDARY] Raw response: '{response[:80]}'")

        if 'different document' in response or 'different documents' in response:
            return False
        elif 'same document' in response:
            return True
        elif 'yes' in response[:50]:
            return True
        elif 'no' in response[:50]:
            return False
        else:
            return False
    except Exception as e:
        print(f"Boundary detection error: {e}")
        return False

## 📑 PDF Processing Pipeline

In [ ]:
# ============================================
# STEP 5: Advanced PDF Processing Pipeline
# ============================================
# WHAT WE'RE DOING:
# Defining extract_and_analyze_pdf(), which opens the uploaded PDF page by
# page, extracts text (with OCR fallback for scanned pages), calls
# classify_document_type() on the first page of each segment, and uses
# detect_document_boundary() to decide where one pharmaceutical document
# ends and the next begins.

# WHY THIS MATTERS:
# A pharmaceutical blob PDF is not a single document -- it is many
# documents concatenated together. This function produces two outputs:
# (1) a list of PageInfo objects (one per page) and (2) a list of
# LogicalDocument objects (one per identified sub-document). Accurate
# boundary detection is critical: if pages are grouped incorrectly,
# retrieved chunks will mix content from unrelated documents.
# ============================================

def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    Supports various file types including scanned PDFs with OCR.
    """
    print("Starting PDF extraction and analysis...")

    # Extract text from each page
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()

        # If no text found, try OCR (for scanned documents)
        if not text.strip():
            print(f"  Page {i}: No text found, attempting OCR...")
            try:
                pix = page.get_pixmap()
                img_data = pix.tobytes("png")
                from PIL import Image
                import pytesseract
                import io
                img = Image.open(io.BytesIO(img_data))
                text = pytesseract.image_to_string(img)
                print(f"  Page {i}: OCR extracted {len(text)} characters")
            except Exception as e:
                print(f"  Page {i}: OCR failed - {e}")
                text = ""

        pages_info.append(PageInfo(page_num=i, text=text))

    doc.close()

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    print(f"Extracted {len(pages_info)} pages")

    # ── Classify ALL pages first ──────────────────────────────────────────
    # Classify every page independently before any boundary detection.
    # This gives us the ground truth type for each page upfront.
    print("Classifying all pages...")
    for i, page_info in enumerate(pages_info):
        page_type = classify_document_type(page_info.text)
        page_info.doc_type = page_type
        print(f"  Page {i}: Classified as {page_type}")

    # ── Boundary detection using classifications ──────────────────────────
    # Now group pages into logical documents. If two consecutive pages have
    # DIFFERENT types, force a new document without calling the LLM.
    # Only call detect_document_boundary when types are the same.
    print("Analyzing document structure...")
    logical_docs = []
    current_doc_pages = [pages_info[0]]
    current_doc_type = pages_info[0].doc_type
    doc_counter = 0

    for i in range(1, len(pages_info)):
        page_info = pages_info[i]
        page_type = page_info.doc_type  # already classified above
        prev_text = pages_info[i-1].text

        if page_type != current_doc_type and page_type != "Other":
            # ── Type changed — force new document, skip LLM call ─────────
            print(f"  Page {i}: [TYPE CHANGE] {current_doc_type} → {page_type}: new document")
            is_same = False
        else:
            # ── Same type — let boundary detection decide ─────────────────
            is_same = detect_document_boundary(
                prev_text,
                page_info.text,
                current_doc_type,
                curr_doc_type=page_type
            )
            print(f"  Page {i}: [BOUNDARY] same={is_same} (type={page_type})")

        if is_same:
            # Continue current document
            page_info.page_in_doc = len(current_doc_pages)
            current_doc_pages.append(page_info)
        else:
            # Save current document and start a new one
            logical_docs.append(LogicalDocument(
                doc_id=f"doc_{doc_counter}",
                doc_type=current_doc_type,
                page_start=current_doc_pages[0].page_num,
                page_end=current_doc_pages[-1].page_num,
                text="\n\n".join([p.text for p in current_doc_pages])
            ))
            doc_counter += 1

            # Start new document
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            current_doc_type = page_type
            print(f"  Page {i}: New document started - {current_doc_type}")

    # Save the last document
    if current_doc_pages:
        logical_docs.append(LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text="\n\n".join([p.text for p in current_doc_pages])
        ))

    print(f"Identified {len(logical_docs)} logical documents")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

    return pages_info, logical_docs

## The following code chunk is for debugging purposes only

In [ ]:
# import fitz

# pdf_path = "/content/pharma-blob-sample.pdf"
# doc = fitz.open(pdf_path)

# prev_doc_type = None
# for i, page in enumerate(doc):
#     text = page.get_text()
#     print(f"\n{'='*50}")
#     print(f"PAGE {i}")
#     print(f"{'='*50}")

#     page_type = classify_document_type(text)
#     print(f"  → Classified as: {page_type}")

#     if i > 0:
#         # Simulate the override logic directly
#         type_changed = (page_type != prev_doc_type and page_type != "Other")
#         if type_changed:
#             print(f"  → [OVERRIDE] Type changed {prev_doc_type} → {page_type}: NEW DOCUMENT")
#         else:
#             prev_text = doc[i-1].get_text()
#             same = detect_document_boundary(prev_text, text, prev_doc_type, curr_doc_type=page_type)
#             print(f"  → Same as previous doc? {same}")

#     prev_doc_type = page_type

## ✂️ Intelligent Chunking with Metadata Preservation

In [ ]:
# ============================================
# STEP 6: Intelligent Chunking with Metadata Preservation
# ============================================
# WHAT WE'RE DOING:
# Breaking each LogicalDocument into smaller overlapping text chunks and
# attaching rich metadata (document type, page range, chunk index) to each
# one. Two approaches are provided: a custom sliding-window chunker and a
# LlamaIndex SentenceSplitter that respects sentence boundaries.
#
# WHY THIS MATTERS:
# Language models and vector search work best with short focused text
# segments rather than entire documents. Overlapping chunks (100-word
# overlap) prevent important information from being split across chunk
# boundaries. Preserving metadata on every chunk ensures that when a chunk
# is retrieved, we can tell the user exactly which document type and page
# it came from -- essential for pharmaceutical compliance traceability.
# ============================================

def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                chunk_size: int = 500,
                                overlap: int = 100) -> List[ChunkMetadata]:
    """
    Chunk a logical document while preserving rich metadata.
    Uses sliding window with overlap for better context.
    """
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        # Document is small enough to be a single chunk
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text
        )
        chunks_metadata.append(chunk_meta)
    else:
        # Create overlapping chunks
        stride = chunk_size - overlap
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            # Calculate which pages this chunk spans
            # (simplified - in production, track more precisely)
            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunk_meta = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text
            )
            chunks_metadata.append(chunk_meta)

            if end_idx >= len(words):
                break

    return chunks_metadata

def chunk_with_llama_index(logical_doc: LogicalDocument,
                           chunk_size: int = 500,
                           chunk_overlap: int = 100) -> List[Document]:
    """
    Alternative: Use LlamaIndex's advanced chunking with metadata.
    """
    # Create LlamaIndex document with metadata
    doc = Document(
        text=logical_doc.text,
        metadata={
            "doc_id": logical_doc.doc_id,
            "doc_type": logical_doc.doc_type,
            "page_start": logical_doc.page_start,
            "page_end": logical_doc.page_end,
            "source": f"{logical_doc.doc_type}_document"
        }
    )

    # Use LlamaIndex's sentence splitter for better chunking
    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator="\n\n",
        separator=" ",
    )

    # Create nodes (chunks) from document
    nodes = splitter.get_nodes_from_documents([doc])

    # Convert to our ChunkMetadata format for consistency
    chunks_metadata = []
    for i, node in enumerate(nodes):
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=i,
            page_start=node.metadata.get("page_start", logical_doc.page_start),
            page_end=node.metadata.get("page_end", logical_doc.page_end),
            text=node.text
        )
        chunks_metadata.append(chunk_meta)

    return chunks_metadata

def process_all_documents(logical_docs: List[LogicalDocument],
                         use_llama_index: bool = False) -> List[ChunkMetadata]:
    """
    Process all logical documents into chunks with metadata.
    Can use either custom or LlamaIndex chunking.
    """
    all_chunks = []

    for logical_doc in logical_docs:
        if use_llama_index:
            chunks = chunk_with_llama_index(logical_doc)
        else:
            chunks = chunk_document_with_metadata(logical_doc)

        logical_doc.chunks = chunks  # Store reference
        all_chunks.extend(chunks)
        print(f"  {logical_doc.doc_type}: Created {len(chunks)} chunks")

    return all_chunks

## 🎯 Query Routing and Intelligent Retrieval

In [ ]:
# ============================================
# STEP 7: Query Routing and Intelligent Retrieval
# ============================================
# WHAT WE'RE DOING:
# Defining predict_query_document_type() which asks Mistral-7B-Instruct model to guess which
# of the 7 pharmaceutical document types is most likely to contain the
# answer to the user's question. Then defining IntelligentRetriever, which
# builds per-document-type FAISS indices and uses the query routing
# prediction to search the most relevant index first.
#
# WHY THIS MATTERS:
# Searching all chunks equally would mix results from a Cover Letter with
# results from a Certificate of Quality. By routing "What is the lot
# number?" to the Certificate Of Quality index and "Who is the supplier?"
# to Supplier Qualification, we improve both precision and speed. When
# routing confidence is low (<0.7) the system falls back to a global search
# across all document types.
# ============================================

def predict_query_document_type(query: str) -> Tuple[str, float]:
    """
    Predict which pharmaceutical document type is most likely to contain
    the answer. Returns predicted type and confidence score.
    """
    prompt = f"""Analyze this query and predict which pharmaceutical document type
would most likely contain the answer.

Query: "{query}"

Choose the MOST LIKELY type from:
- Cover Letter: Formal letters about product information or storage conditions
- Certificate Of Quality: Lot numbers, manufacture/expiration dates, test results
- Packaging Specification: Packaging components, materials, part numbers
- Bse/Tse Declaration: Animal-origin material declarations, TSE compliance
- Material Description: Materials of construction, sterilization compatibility
- Supplier Qualification: Supplier audits, ISO certifications, approved products
- Chain Of Custody: Manufactured assemblies, traceability, shipment flow
- Other: General or unclear queries

Respond in JSON format:
{{"type": "DocumentType", "confidence": 0.85}}

Confidence should be between 0.0 and 1.0"""

    try:
        response = str(llm.complete(prompt)).strip()
        # Strip any text before/after the JSON
        import re
        json_match = re.search(r'\{.*?\}', response, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
        else:
            return "Other", 0.0
        predicted = result.get("type", "Other")
        confidence = result.get("confidence", 0.5)
        return clean_doc_type(predicted), confidence
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0

class IntelligentRetriever:
    """
    Advanced retrieval system with metadata filtering and query routing.
    """

    def __init__(self):
        self.index = None
        self.chunks_metadata = []
        self.doc_type_indices = {}  # Separate indices per doc type

    def build_indices(self, chunks_metadata: List[ChunkMetadata]):
        """
        Build FAISS indices with document type segregation.
        """
        print("Building vector indices...")
        self.chunks_metadata = chunks_metadata

        # Create embeddings for all chunks
        texts = [chunk.text for chunk in chunks_metadata]
        embeddings = np.array(llama_embed_model.get_text_embedding_batch(texts), dtype=np.float32)


        # Store embeddings in metadata
        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        # Build main index
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

        # Build separate indices for each document type
        doc_types = set(chunk.doc_type for chunk in chunks_metadata)
        for doc_type in doc_types:
            type_indices = [i for i, chunk in enumerate(chunks_metadata)
                          if chunk.doc_type == doc_type]
            if type_indices:
                type_embeddings = embeddings[type_indices]
                type_index = faiss.IndexFlatL2(dim)
                type_index.add(type_embeddings)
                self.doc_type_indices[doc_type] = {
                    'index': type_index,
                    'mapping': type_indices  # Maps back to original chunks
                }

        print(f"Indexed {len(chunks_metadata)} chunks across {len(doc_types)} document types")

    def retrieve(self, query: str, k: int = 4,
                filter_doc_type: Optional[str] = None,
                auto_route: bool = True) -> List[Tuple[ChunkMetadata, float]]:
        """
        Retrieve relevant chunks with optional filtering and routing.
        Returns chunks with relevance scores.
        """
        query_embedding = np.array([llama_embed_model.get_query_embedding(query)], dtype=np.float32)

        # Determine which index to search
        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            # Use filtered index
            type_data = self.doc_type_indices[filter_doc_type]
            D, I = type_data['index'].search(query_embedding, k)
            # Map back to original chunks
            chunk_indices = [type_data['mapping'][i] for i in I[0]]
            distances = D[0]
        elif auto_route:
            # Predict best document type
            predicted_type, confidence = predict_query_document_type(query)
            print(f"Query routed to: {predicted_type} (confidence: {confidence:.2f})")

            if confidence > 0.7 and predicted_type in self.doc_type_indices:
                # High confidence - use specific index
                type_data = self.doc_type_indices[predicted_type]
                D, I = type_data['index'].search(query_embedding, k)
                chunk_indices = [type_data['mapping'][i] for i in I[0]]
                distances = D[0]
            else:
                # Low confidence - search all
                D, I = self.index.search(query_embedding, k)
                chunk_indices = I[0]
                distances = D[0]
        else:
            # Search all chunks
            D, I = self.index.search(query_embedding, k)
            chunk_indices = I[0]
            distances = D[0]

        # Convert distances to similarity scores (inverse)
        max_dist = max(distances) if len(distances) > 0 else 1.0
        scores = [(max_dist - d) / max_dist for d in distances]

        results = [(self.chunks_metadata[i], scores[idx])
                  for idx, i in enumerate(chunk_indices)]

        return results

## 💬 Answer Generation with Source Attribution

In [ ]:
# ============================================
# STEP 8: Answer Generation with Source Attribution
# ============================================
# WHAT IS HAPPENING:
# Defining generate_answer_with_sources(), which takes the top-k retrieved
# chunks and sends them to Mistral as context, asking it to answer the user's
# question using only that context. The function returns the answer text, a
# list of source citations (document type, page range, relevance score),
# and an overall confidence score derived from the retrieval scores.
#
# WHY THIS MATTERS:
# Pharmaceutical documentation must be traceable. Simply generating an
# answer without citing which document and page it came from would not meet
# compliance standards. By injecting "[From Certificate Of Quality, Pages
# 2-3]" labels into the context prompt, we encourage Mistral to ground its
# answer in the actual source material and to tell the user where to verify
# the information.
# ============================================

def generate_answer_with_sources(query: str,
                                retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """
    Generate answer with detailed source attribution.
    """
    if not retrieved_chunks:
        return {
            'answer': "I couldn't find relevant information to answer your question.",
            'sources': [],
            'confidence': 0.0
        }

    # Prepare context from retrieved chunks
    context_parts = []
    sources = []

    for chunk_meta, score in retrieved_chunks:
        context_parts.append(f"[From {chunk_meta.doc_type}, Pages {chunk_meta.page_start}-{chunk_meta.page_end}]")
        context_parts.append(chunk_meta.text)
        context_parts.append("")

        sources.append({
            'doc_type': chunk_meta.doc_type,
            'pages': f"{chunk_meta.page_start}-{chunk_meta.page_end}",
            'relevance': f"{score:.2%}",
            'preview': chunk_meta.text[:100] + "..."
        })

    context = "\n".join(context_parts)

    # Generate answer
    prompt = f"""You are answering questions about pharmaceutical documentation
including certificates of quality, packaging specifications, and compliance
declarations. Use the provided context to answer the question accurately.
Be specific and cite which document type and pages support your answer.

Context:
{context}

Question: {query}

Instructions:
1. Answer based ONLY on the provided context
2. Mention which document type(s) contain the information
3. Be concise but complete
4. If the context doesn't contain enough information, say so

Answer:"""

    try:
        #Mistral
        response = llm.complete(prompt)
        answer = str(response).strip()

        # Calculate overall confidence based on retrieval scores
        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)

        return {
            'answer': answer,
            'sources': sources,
            'confidence': avg_score,
            'chunks_used': len(retrieved_chunks)
        }
    except Exception as e:
        print(f"Answer generation error: {e}")
        return {
            'answer': f"Error generating answer: {str(e)}",
            'sources': sources,
            'confidence': 0.0
        }

## 🏗️ Document Store

In [ ]:
# ============================================
# STEP 9: Document Store
# ============================================
# WHAT IS HAPPENING:
# Defining EnhancedDocumentStore, the central orchestrator class that ties
# together all previous steps: PDF extraction, logical document grouping,
# chunking, index building, and query handling. A single global instance
# (doc_store) is created and shared across the Gradio UI callbacks.
#
# WHY THIS MATTERS:
# The document store acts as the in-memory database for the entire session.
# It holds all page data, logical documents, chunk metadata, and the FAISS
# indices. Every Gradio interaction (upload, question, clear) goes through
# this class, which keeps state management simple and avoids re-processing
# the PDF on every query.
# ============================================

class EnhancedDocumentStore:
    """
    Manages the complete document processing and retrieval pipeline.
    """

    def __init__(self):
        self.pages_info = []
        self.logical_docs = []
        self.chunks_metadata = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.filename = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        """
        Complete PDF processing pipeline.
        """
        self.filename = filename
        self.is_ready = False
        start_time = datetime.now()

        try:
            # Extract and analyze PDF
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)

            # Chunk documents with metadata
            self.chunks_metadata = process_all_documents(self.logical_docs)

            # Build retrieval indices
            self.retriever.build_indices(self.chunks_metadata)

            # Calculate processing statistics
            process_time = (datetime.now() - start_time).total_seconds()
            self.processing_stats = {
                'filename': filename,
                'total_pages': len(self.pages_info),
                'documents_found': len(self.logical_docs),
                'total_chunks': len(self.chunks_metadata),
                'document_types': list(set(doc.doc_type for doc in self.logical_docs)),
                'processing_time': f"{process_time:.1f}s"
            }

            self.is_ready = True
            return True, self.processing_stats

        except Exception as e:
            return False, {'error': str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
             auto_route: bool = True, k: int = 4) -> Dict:
        """
        Query the document store.
        """
        if not self.is_ready:
            return {
                'answer': "Please upload and process a PDF first.",
                'sources': [],
                'confidence': 0.0
            }

        # Retrieve relevant chunks
        retrieved = self.retriever.retrieve(
            question, k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route
        )

        # Generate answer with sources
        result = generate_answer_with_sources(question, retrieved)
        result['filter_used'] = filter_type or ('auto' if auto_route else 'none')

        return result

    def get_document_structure(self) -> List[Dict]:
        """
        Get the document structure for UI display.
        """
        if not self.logical_docs:
            return []

        structure = []
        for doc in self.logical_docs:
            structure.append({
                'id': doc.doc_id,
                'type': doc.doc_type,
                'pages': f"{doc.page_start + 1}-{doc.page_end + 1}",  # 1-indexed for UI
                'chunks': len(doc.chunks) if doc.chunks else 0,
                'preview': doc.text[:200] + "..." if len(doc.text) > 200 else doc.text
            })

        return structure

## 🎨 A Sophisticated Gradio Interface


In [ ]:
# ============================================
# STEP 10: Gradio Interface
# ============================================
# WHAT IS HAPPENING:
# Building the Gradio web UI that ties all pipeline components together.
# The interface has three columns: a PDF upload on the left, document info
# and retrieval settings in the middle, and a chat panel on the right.
# Users upload a pharmaceutical PDF, the system processes it, detects
# document boundaries, builds the vector index, and then allows natural
# language Q&A against the identified pharmaceutical sub-documents.
# The Pfizer logo is loaded from a local file uploaded to the Colab environment.
#
# WHY THIS MATTERS:
# The Gradio interface makes the RAG pipeline accessible to non-technical
# users. The document type filter dropdown and auto-route toggle give
# advanced users fine-grained control over retrieval scope.
# ============================================

from datetime import datetime
import gradio as gr

# ── Pfizer Branding ───────────────────────────────────────────────────────────
PFIZER_BLUE = "#0057A8"  # matched to the darker blue on your actual logo
PFIZER_LOGO_PATH = "/content/pfizer_logo.png"

CUSTOM_CSS = f"""
/* Primary buttons — Pfizer blue */
button.primary {{
    background-color: {PFIZER_BLUE} !important;
    border-color: {PFIZER_BLUE} !important;
    color: white !important;
}}
/* Checkbox — Pfizer blue */
input[type=checkbox] {{
    accent-color: {PFIZER_BLUE} !important;
}}
/* Slider — Pfizer blue */
input[type=range] {{
    accent-color: {PFIZER_BLUE} !important;
}}
/* Bot bubble left border */
.message.bot .bubble-wrap, .message.assistant .bubble-wrap {{
    border-left: 3px solid {PFIZER_BLUE} !important;
}}
/* User bubble */
.message.user .bubble-wrap {{
    background: #e8f0fa !important;
}}
"""
# ── Global Store ─────────────────────────────────────────────────────────────
doc_store = EnhancedDocumentStore()

# ── Handler Functions ─────────────────────────────────────────────────────────
def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return "Please upload a PDF file.", None, gr.update(choices=["All"])

    success, stats = doc_store.process_pdf(
        pdf_file,
        filename=pdf_file.split('/')[-1] if isinstance(pdf_file, str) else
                 getattr(pdf_file, 'name', 'document.pdf')
    )

    if success:
        status_msg = f"""
**Successfully Processed:**
- File: {stats['filename']}
- Pages: {stats['total_pages']}
- Documents Found: {stats['documents_found']}
- Chunks Created: {stats['total_chunks']}
- Types: {', '.join(stats['document_types'])}
- Time: {stats['processing_time']}
"""
        structure = doc_store.get_document_structure()
        structure_display = "\n".join([
            f"- **{doc['type']}** (Pages {doc['pages']}): {doc['chunks']} chunks"
            for doc in structure
        ])
        doc_types = ["All"] + stats['document_types']
        return status_msg, structure_display, gr.update(choices=doc_types, value="All")
    else:
        return f"Error: {stats.get('error', 'Unknown error')}", None, gr.update(choices=["All"])

def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Handle chat interactions."""
    if not doc_store.is_ready:
        response = "Please upload and process a pharmaceutical PDF document first."
        return history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": response}
        ]

    filter_type = None if doc_filter == "All" else doc_filter
    result = doc_store.query(
        message,
        filter_type=filter_type,
        auto_route=auto_route and filter_type is None,
        k=num_chunks
    )

    response = f"{result['answer']}\n\n"
    if result['sources']:
        response += "**Sources:**\n"
        for src in result['sources']:
            response += f"- {src['doc_type']} (Pages {src['pages']}) - Relevance: {src['relevance']}\n"
    response += f"\n*Confidence: {result['confidence']:.1%} | Filter: {result['filter_used']}*"

    return history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

def save_chat(history):
    """Save chat history to a text file and return path for download."""
    filename = f"/content/chat_history_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(filename, "w") as f:
        for msg in history:
            role = msg.get("role", "").capitalize()
            content = msg.get("content", "")
            f.write(f"{role}: {content}\n")
            f.write("-" * 40 + "\n")
    return filename

# ── Interface Builder ─────────────────────────────────────────────────────────
def create_interface():

    # Load logo fresh each time — works after every Colab reconnect
    import base64, os
    if os.path.exists(PFIZER_LOGO_PATH):
        with open(PFIZER_LOGO_PATH, "rb") as f:
            logo_b64 = base64.b64encode(f.read()).decode("utf-8")
        logo_src = f"data:image/png;base64,{logo_b64}"
    else:
        logo_src = ""  # blank if not uploaded yet

    with gr.Blocks(title="Pfizer Pharmaceutical Document Q&A") as demo:

        # ── Header ───────────────────────────────────────────────────────────
        gr.HTML(f"""
            <div style="
                display: flex;
                flex-direction: row;
                align-items: center;
                gap: 20px;
                padding: 12px 0 8px 0;
                border-bottom: 2px solid {PFIZER_BLUE};
                margin-bottom: 12px;
            ">
                <img src="{logo_src}"
                     style="height: 60px; width: auto; flex-shrink: 0;"
                     alt="Pfizer Logo">
                <span style="
                    font-size: 15px;
                    color: #333;
                    font-weight: 500;
                    line-height: 1.4;
                ">
                    Pharmaceutical Document Q&amp;A System —
                    Intelligent Multi-Document Analysis with Advanced RAG Pipeline
                </span>
            </div>
        """)

        # ── Main Layout ───────────────────────────────────────────────────────
        with gr.Row():

            # ── LEFT: PDF Upload ──────────────────────────────────────────────
            with gr.Column(scale=2):
                gr.Markdown("### 📄 Document Upload")
                pdf_input = gr.File(
                    label="Upload Pharmaceutical PDF",
                    file_types=[".pdf"],
                    type="filepath"
                )
                with gr.Row():
                    process_btn = gr.Button(
                        "🔄 Process Document",
                        variant="primary",
                        size="lg",
                        scale=2
                    )
                    clear_all_btn = gr.Button(
                        "🗑️ Clear All",
                        variant="secondary",
                        size="lg",
                        scale=1
                    )

                gr.Markdown("### 📊 Document Info")
                status_output = gr.Markdown(value="Waiting for PDF upload...")
                structure_output = gr.Markdown(value="")

            # ── MIDDLE: Settings ──────────────────────────────────────────────
            with gr.Column(scale=1):
                gr.Markdown("### ⚙️ Retrieval Settings")
                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Filter search to a specific pharmaceutical document type"
                )
                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect the most relevant document type"
                )
                num_chunks = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=4,
                    step=1,
                    label="Chunks to Retrieve"
                )

                gr.Markdown("### 💾 Chat Export")
                save_btn = gr.Button("💾 Save Chat", variant="secondary")
                download_file = gr.File(label="Download Chat History")

            # ── RIGHT: Chat ───────────────────────────────────────────────────
            with gr.Column(scale=2):
                gr.Markdown("### 💬 Ask Questions")
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=500,
                    show_label=False,
                    avatar_images=(
                        None,               # user — default
                        PFIZER_LOGO_PATH    # assistant — Pfizer logo
                    )
                )
                with gr.Row():
                    msg_input = gr.Textbox(
                        placeholder="e.g., What is the lot number? What sterilization method was used?",
                        scale=4,
                        show_label=False
                    )
                    send_btn = gr.Button("📤 Send", scale=1, variant="primary")

                with gr.Row():
                    clear_chat_btn = gr.Button("🗑️ Clear Chat", size="sm", scale=1)
                    example_btn1 = gr.Button("📋 Summarise document", size="sm", scale=1)
                    example_btn2 = gr.Button("🔢 Find lot numbers", size="sm", scale=1)

        # ── Status Bar ────────────────────────────────────────────────────────
        with gr.Row():
            status_bar = gr.Markdown(
                value="**Status:** Ready | **Documents:** 0 | **Chunks:** 0"
            )

        # ── Helper Functions ──────────────────────────────────────────────────
        def update_status_bar():
            if doc_store.is_ready:
                stats = doc_store.processing_stats
                return (
                    f"**Status:** Ready | "
                    f"**Documents:** {stats.get('documents_found', 0)} | "
                    f"**Chunks:** {stats.get('total_chunks', 0)}"
                )
            return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"

        def clear_all():
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (
                None,
                "Waiting for PDF upload...",
                "",
                gr.update(choices=["All"], value="All"),
                [],
                "",
                update_status_bar()
            )

        def process_pdf_with_status(pdf_file):
            status, structure, filter_update = process_pdf_handler(pdf_file)
            return status, structure, filter_update, update_status_bar()

        def chat_with_status(message, history, doc_filter, auto_route, num_chunks):
            new_history = chat_handler(message, history, doc_filter, auto_route, num_chunks)
            return new_history, update_status_bar()

        def ask_summary(history):
            return chat_handler(
                "Can you provide a summary of the main points in this document?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        def ask_lot_numbers(history):
            return chat_handler(
                "What lot numbers or batch numbers are mentioned in these documents?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        # ── Event Wiring ──────────────────────────────────────────────────────
        process_btn.click(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output,
                     doc_filter, chatbot, msg_input, status_bar]
        )

        msg_input.submit(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(lambda: "", outputs=[msg_input])

        send_btn.click(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(lambda: "", outputs=[msg_input])

        clear_chat_btn.click(lambda: [], outputs=[chatbot])

        save_btn.click(fn=save_chat, inputs=[chatbot], outputs=[download_file])

        example_btn1.click(
            fn=ask_summary,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(fn=update_status_bar, outputs=[status_bar])

        example_btn2.click(
            fn=ask_lot_numbers,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(fn=update_status_bar, outputs=[status_bar])

    return demo


In [ ]:
# ============================================
# STEP 11: Launch the Application
# ============================================
# WHAT IS HAPPENING:
# Creating the Gradio interface instance and launching it. The share=True
# flag generates a public temporary URL so anyone with the link can access
# the running app from outside Colab. debug=True prints server-side logs
# to the cell output so you can see what happens when the PDF is processed
# and when questions are asked.
#
# WHY THIS MATTERS:
# This is the entry point that makes everything visible and interactive.
# Without this cell, all the pipeline code above would be defined but
# nothing would be rendered for the user. Once launched, the app stays
# alive until you stop the cell or the Colab session times out.
# A public Gradio URL (https://....gradio.live) and an inline iframe
# showing the Pharmaceutical Document Q&A System. Open the URL or use the
# iframe, upload pharma-blob-sample.pdf, and start asking questions.
# ============================================

# ── Launch ────────────────────────────────────────────────────────────────────
demo = create_interface()
demo.launch(share=True, debug=False, css=CUSTOM_CSS)

## Test Cell

In [ ]:
import time

test_questions = [
    {
        "question": "What is the material description for this product?",
        "expected_doc_type": "Material Description",
        "expected_pages": "6"
    },
    {
        "question": "What are the part numbers listed in this document?",
        "expected_doc_type": "Packaging Specification",
        "expected_pages": "3"
    },
    {
        "question": "Were there any packaging configuration changes?",
        "expected_doc_type": "Packaging Specification",
        "expected_pages": "3"
    },
    {
        "question": "What are the BSE/TSE compliance dates?",
        "expected_doc_type": "Bse/Tse Declaration",
        "expected_pages": "5"
    },
    {
        "question": "What supplier information is available?",
        "expected_doc_type": "Supplier Qualification",
        "expected_pages": "7"
    },
]

results = []

for test in test_questions:
    start = time.time()
    result = doc_store.query(test["question"], auto_route=True, k=4)
    elapsed = time.time() - start

    # Check sources for correct doc type and pages
    sources_str = str(result["sources"]).lower()
    correct_doc  = test["expected_doc_type"].lower() in sources_str
    correct_pages = test["expected_pages"] in sources_str

    results.append({
        "question":      test["question"],
        "correct_doc":   correct_doc,
        "correct_pages": correct_pages,
        "response_time": elapsed,
        "confidence":    result.get("confidence", 0),
    })

    print(f"\nQ: {test['question']}")
    print(f"  Sources:        {result['sources']}")
    print(f"  Doc type correct:  {correct_doc}")
    print(f"  Pages correct:     {correct_pages}")
    print(f"  Confidence:        {result.get('confidence', 0):.1%}")
    print(f"  Response time:     {elapsed:.2f}s")

# ── Summary ───────────────────────────────────────────────────────────────────
hit_rate  = sum(r["correct_doc"]   for r in results) / len(results) * 100
page_acc  = sum(r["correct_pages"] for r in results) / len(results) * 100
avg_time  = sum(r["response_time"] for r in results) / len(results)
avg_conf  = sum(r["confidence"]    for r in results) / len(results)

print(f"\n{'='*45}")
print(f"  Hit Rate (correct doc type): {hit_rate:.1f}%")
print(f"  Citation Accuracy (pages):   {page_acc:.1f}%")
print(f"  Avg Response Time:           {avg_time:.2f}s")
print(f"  Avg Confidence:              {avg_conf:.1%}")
print(f"  Test Questions Run:          {len(results)}")